In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/README.md
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/manifest.csv
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/manifest.csv.sha256
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/manifest.json
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/manifest.json.sha256
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/test/glioma/brisc2025_test_00001_gl_ax_t1.jpg
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/test/glioma/brisc2025_test_00002_gl_ax_t1.jpg
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/test/glioma/brisc2025_test_00003_gl_ax_t1.jpg
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/test/glioma/brisc2025_test_00004_gl_ax_t1.jpg
/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/test/glioma/brisc2025_test_00005_gl_ax_t1.jpg
/kaggle/input/datasets/briscdata

In [2]:
"""
WaveSSM-TriNet (revised) -- Kaggle T4 pipeline, BRISC 2025 (+ Nickparvar external)
================================================================================
Changes versus the first submission, keyed to Reviewer-1 comment numbers:
  #12/#13  TCAFG now attends over TOKEN SEQUENCES (49 tokens/branch), six directional
           multi-head cross-attentions, a gate per pair, attention-entropy logging.
  #20      Capacity/fusion baselines: concat-MLP, standard MHA (transformer) fusion, gated
           fusion, EfficientNet-only (B3 / B4); branch-subset ablations. See VARIANTS.
  #21      Multi-seed runs, mean/SD/95% CI, paired tests (EVALUATION UTILITIES section).
  #3/#4/#7 Near-duplicate detection, duplicate clusters used as groups, group-aware
           stratified train/val split, group-aware CV. Official BRISC test split is
           NEVER used for checkpoint selection (it is scored once).
  #5/#6    Every table/figure is generated from saved prediction CSVs (EVALUATION UTILITIES section).
  #15      Evidential loss: float32, logit clamp, max-evidence cap, hard labels in the KL
           term, configurable label smoothing and KL annealing. Loss is the expected-MSE
           (Brier-type) form -- the paper text must say so, not "cross-entropy".
  #16-#18  ECE/Brier/NLL, reliability, temperature scaling on VAL, error-detection AUROC,
           risk-coverage, high-confidence-error table.
  #22      Branch occlusion (mean-token substitution) + exact 3-player Shapley values.
  #23      Grad-CAM for ALL three branches w.r.t. the full-model decision (+ optional
           mask hit-rate if BRISC masks are available).
  #24      Stratified t-SNE (+UMAP if installed).
  #25/#28  Intensity/contrast shift statistics, external per-class uncertainty.
  #26      FlopCounterMode (counts F.conv2d and attention matmuls) + analytic scan FLOPs +
           measured latency / throughput / peak memory.
  #27      Exact protocol dumped to run_config.json; LR trajectory saved; pretrained
           weights REQUIRED (no silent fallback to random init / other backbones).
  #8       Haar DWT unit-tested against PyWavelets.
  #29      Split manifests, predictions, configs, environment saved to OUT_DIR.

SINGLE-FILE, KAGGLE GPU T4 x2 VERSION (eval_utils.py merged in).  Paste into ONE notebook cell and run.
  * Two worker processes, one per T4 (CUDA_VISIBLE_DEVICES), pull (variant, seed) jobs from a shared queue ->
    ~2x throughput with the SAME protocol (batch size, seeds, epochs) as a single-GPU run. No DataParallel.
  * A wall-clock budget (CFG["time_budget_h"], default 11.5 h) decides which jobs still fit; jobs are ordered by priority,
    so what is dropped (if anything) is the least important. Finished runs are never retrained (resumable).
  * RUN_MODE: "paper" (default: 5 seeds x 50 ep + 5-fold CV) | "quick" (3 seeds x 30 ep) | "smoke" (2-min synthetic test).
  * Do not put %%magics or other cells' code in the same cell; keep Internet ON (timm weights) and Accelerator = GPU T4 x2.
"""
# WAVESSM-TRINET-SCRIPT
import os, sys, json, math, time, copy, random, warnings, subprocess, traceback
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve, auc,
                             cohen_kappa_score, matthews_corrcoef,
                             precision_recall_fscore_support, average_precision_score)
from sklearn.preprocessing import label_binarize
import seaborn as sns


warnings.filterwarnings("ignore")
SMOKE = os.environ.get("SMOKE", "0") == "1" or os.environ.get("RUN_MODE", "") == "smoke"

# ══════════════════════════════════════════════════════════════════════════════
#  EVALUATION UTILITIES  (merged from eval_utils.py -- no torch dependency)
# ══════════════════════════════════════════════════════════════════════════════
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
PALETTE = ["#1F6FEB", "#D9480F", "#1A7F37", "#B08800", "#8250DF"]


# ══════════════════════════════════════════════════════════════════════════════
#  1. DATA HYGIENE: near-duplicate detection, clusters, group-aware splits
#     (Reviewer comments 2, 3, 4, 7)
# ══════════════════════════════════════════════════════════════════════════════
def thumbnail_matrix(paths, size=32, cache=None):
    """Mean-centred, L2-normalised grey thumbnails (N, size*size). Cosine = dot product."""
    from PIL import Image
    if cache and os.path.exists(cache):
        arr = np.load(cache)
        if arr.shape[0] == len(paths):
            return arr
    out = np.zeros((len(paths), size * size), dtype=np.float32)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            a = np.asarray(im.convert("L").resize((size, size), Image.BILINEAR), dtype=np.float32).ravel()
        a = a - a.mean()
        n = np.linalg.norm(a)
        out[i] = a / n if n > 1e-6 else 0.0
    if cache:
        np.save(cache, out)
    return out


def cosine_pairs(A, B=None, thr=0.985, chunk=1024):
    """All index pairs with cosine >= thr. If B is None, pairs (i<j) inside A."""
    same = B is None
    B = A if same else B
    ii_all, jj_all, ss_all = [], [], []
    for s in range(0, len(A), chunk):
        sim = A[s:s + chunk] @ B.T
        ii, jj = np.nonzero(sim >= thr)
        gi = ii + s
        keep = jj > gi if same else np.ones_like(jj, dtype=bool)
        ii_all.append(gi[keep]); jj_all.append(jj[keep]); ss_all.append(sim[ii, jj][keep])
    if not ii_all:
        return np.zeros((0, 3))
    return np.stack([np.concatenate(ii_all), np.concatenate(jj_all), np.concatenate(ss_all)], 1) \
        if ii_all else np.zeros((0, 3))


def cluster_from_pairs(n, pairs):
    """Union-find. Returns an integer cluster id per item (near-duplicate groups)."""
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j, _ in pairs:
        ri, rj = find(int(i)), find(int(j))
        if ri != rj:
            parent[rj] = ri
    roots = np.array([find(i) for i in range(n)])
    _, cl = np.unique(roots, return_inverse=True)
    return cl


def group_stratified_split(labels, groups, val_frac=0.1, seed=42):
    """One stratified, group-aware split. Returns (train_idx, val_idx)."""
    n_splits = max(2, int(round(1.0 / val_frac)))
    labels, groups = np.asarray(labels), np.asarray(groups)
    min_groups = min(len(np.unique(groups[labels == c])) for c in np.unique(labels))   # robust to tiny / unbalanced data
    n_splits = max(2, min(n_splits, min_groups))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr, va = next(sgkf.split(np.zeros(len(labels)), labels, groups))
    return tr, va


def image_intensity_stats(paths):
    """Cheap acquisition-shift descriptors (comment 25): mean, contrast, background fraction."""
    from PIL import Image
    rows = []
    for p in paths:
        with Image.open(p) as im:
            a = np.asarray(im.convert("L").resize((128, 128)), dtype=np.float32) / 255.0
        fg = a[a > 0.05]
        rows.append(dict(path=p, mean=float(a.mean()), std=float(a.std()),
                         bg_frac=float((a <= 0.05).mean()),
                         fg_mean=float(fg.mean()) if fg.size else 0.0,
                         p95=float(np.percentile(a, 95))))
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
#  2. PREDICTION-FILE HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def arrays_from_df(df, K=4):
    y = df["label"].values.astype(int)
    pred = df["pred"].values.astype(int)
    prob = df[[f"p{k}" for k in range(K)]].values.astype(np.float64)
    vac = df["vacuity"].values.astype(np.float64) if "vacuity" in df else np.full(len(df), np.nan)
    return y, pred, prob, vac


def predictive_entropy(prob):
    K = prob.shape[1]
    return -(prob * np.log(np.clip(prob, 1e-12, 1))).sum(1) / math.log(K)


# ══════════════════════════════════════════════════════════════════════════════
#  3. CLASSIFICATION METRICS + CONFIDENCE INTERVALS   (comments 5, 6, 21)
# ══════════════════════════════════════════════════════════════════════════════
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a) / d, (c + a) / d)


def mean_auc_ovr(y, prob):
    K = prob.shape[1]
    lb = label_binarize(y, classes=list(range(K)))
    out = []
    for k in range(K):
        if lb[:, k].sum() in (0, len(y)):
            out.append(float("nan")); continue
        out.append(roc_auc_score(lb[:, k], prob[:, k]))
    return out


def classification_metrics(y, pred, prob, class_names=CLASS_NAMES):
    K = len(class_names)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=list(range(K)), zero_division=0)
    _, _, wf1, _ = precision_recall_fscore_support(y, pred, labels=list(range(K)), average="weighted", zero_division=0)
    per_auc = mean_auc_ovr(y, prob)
    k_ok = int((y == pred).sum()); n = len(y)
    lo, hi = wilson_ci(k_ok, n)
    return dict(
        n=n, correct=k_ok, accuracy=k_ok / n, accuracy_ci95=[lo, hi],
        macro_precision=float(pr.mean()), macro_recall=float(rc.mean()),
        macro_f1=float(f1.mean()), weighted_f1=float(wf1),
        mcc=float(matthews_corrcoef(y, pred)), kappa=float(cohen_kappa_score(y, pred)),
        mean_auc=float(np.nanmean(per_auc)),
        per_class={c: dict(precision=float(pr[i]), recall=float(rc[i]), f1=float(f1[i]),
                           support=int(sup[i]), auc=float(per_auc[i]))
                   for i, c in enumerate(class_names)},
        confusion_matrix=confusion_matrix(y, pred, labels=list(range(K))).tolist())


def bootstrap_ci(y, pred, prob, n_boot=1000, seed=0):
    """Percentile bootstrap CIs for accuracy, macro-F1 and mean AUC (substitute for DeLong)."""
    rng = np.random.RandomState(seed)
    accs, f1s, aucs = [], [], []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yy, pp, pr = y[idx], pred[idx], prob[idx]
        accs.append((yy == pp).mean())
        f1s.append(precision_recall_fscore_support(yy, pp, average="macro", zero_division=0)[2])
        aucs.append(np.nanmean(mean_auc_ovr(yy, pr)))
    q = lambda v: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]
    return dict(accuracy=q(accs), macro_f1=q(f1s), mean_auc=q(aucs))


# ══════════════════════════════════════════════════════════════════════════════
#  4. CALIBRATION  (comment 16)
# ══════════════════════════════════════════════════════════════════════════════
def ece_score(prob, y, n_bins=15):
    conf = prob.max(1); pred = prob.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def brier_score(prob, y):
    K = prob.shape[1]
    oh = np.eye(K)[y]
    return float(((prob - oh) ** 2).sum(1).mean())


def nll_score(prob, y):
    return float(-np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean())


def reliability_curve(prob, y, n_bins=15):
    conf = prob.max(1); acc = (prob.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            xs.append(conf[m].mean()); ys.append(acc[m].mean()); ns.append(int(m.sum()))
    return np.array(xs), np.array(ys), np.array(ns)


def apply_temperature(prob, T):
    lg = np.log(np.clip(prob, 1e-12, 1)) / T
    lg = lg - lg.max(1, keepdims=True)
    e = np.exp(lg)
    return e / e.sum(1, keepdims=True)


def fit_temperature(prob, y, grid=np.linspace(0.25, 6.0, 231)):
    """Temperature scaling on log-probabilities (works for softmax and Dirichlet-mean probs).
    MUST be fitted on the validation split, never on the test split."""
    nlls = [nll_score(apply_temperature(prob, T), y) for T in grid]
    return float(grid[int(np.argmin(nlls))])


def calibration_report(prob, y, n_bins=15):
    return dict(ece=ece_score(prob, y, n_bins), brier=brier_score(prob, y), nll=nll_score(prob, y))


# ══════════════════════════════════════════════════════════════════════════════
#  5. UNCERTAINTY QUALITY  (comments 17, 18, 25)
# ══════════════════════════════════════════════════════════════════════════════
def risk_coverage(uncertainty, correct):
    """Selective-risk curve: keep the most-certain fraction, report error rate among kept."""
    order = np.argsort(uncertainty, kind="mergesort")          # most certain first
    err = (~correct[order]).astype(float)
    n = len(err)
    cum_err = np.cumsum(err)
    cov = np.arange(1, n + 1) / n
    risk = cum_err / np.arange(1, n + 1)
    return cov, risk, float(risk.mean())                         # AURC = mean selective risk


def error_detection(y, pred, prob, vac):
    correct = (y == pred)
    scores = {"1-maxprob": 1 - prob.max(1), "entropy": predictive_entropy(prob)}
    if np.isfinite(vac).any():
        scores = {"vacuity": vac, **scores}
    out = {}
    for name, s in scores.items():
        if correct.all() or (~correct).all():
            out[name] = dict(auroc=float("nan"), auprc=float("nan"), aurc=float("nan"))
            continue
        cov, risk, aurc = risk_coverage(s, correct)
        out[name] = dict(auroc=float(roc_auc_score(~correct, s)),
                         auprc=float(average_precision_score(~correct, s)),
                         aurc=aurc,
                         risk_at_90cov=float(risk[int(0.9 * len(risk)) - 1]),
                         risk_at_80cov=float(risk[int(0.8 * len(risk)) - 1]))
    out["_error_rate"] = float((~correct).mean())
    return out


def high_confidence_errors(df, K=4, conf_thr=0.85):
    """Table for comment 18: confident mistakes and where each uncertainty score ranks them."""
    y, pred, prob, vac = arrays_from_df(df, K)
    mp = prob.max(1); ent = predictive_entropy(prob)
    rk = lambda v: pd.Series(v).rank(pct=True).values        # percentile; high = flagged uncertain
    t = pd.DataFrame(dict(path=df["path"].values, label=y, pred=pred, maxprob=mp, entropy=ent,
                          vacuity=vac, entropy_pctile=rk(ent), maxprob_pctile=rk(-mp),
                          vacuity_pctile=rk(vac) if np.isfinite(vac).any() else np.nan))
    return t[(y != pred) & (mp >= conf_thr)].sort_values("maxprob", ascending=False)


# ══════════════════════════════════════════════════════════════════════════════
#  6. STATISTICS ACROSS SEEDS / PAIRED TESTS  (comment 21)
# ══════════════════════════════════════════════════════════════════════════════
def summarize_seeds(vals):
    v = np.asarray(vals, dtype=float)
    n = len(v)
    m = float(v.mean()); sd = float(v.std(ddof=1)) if n > 1 else float("nan")
    ci = float(stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)) if n > 1 else float("nan")
    return dict(n=n, mean=m, sd=sd, ci95=[m - ci, m + ci])


def paired_seed_tests(a, b):
    """a, b: per-seed metric values of two variants (same seeds). Low power at n=5; say so."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    out = dict(mean_diff=float((a - b).mean()))
    if len(a) >= 2 and np.any(a != b):
        out["paired_t_p"] = float(stats.ttest_rel(a, b).pvalue)
        try:
            out["wilcoxon_p"] = float(stats.wilcoxon(a, b).pvalue)
        except ValueError:
            out["wilcoxon_p"] = float("nan")
    return out


def mcnemar_exact(correct_a, correct_b):
    b = int((correct_a & ~correct_b).sum()); c = int((~correct_a & correct_b).sum())
    p = 1.0 if b + c == 0 else float(stats.binomtest(b, b + c, 0.5).pvalue)
    return dict(a_only_correct=b, b_only_correct=c, p=p)


def shapley_values(values, players):
    """values: {frozenset(subset): v(subset)}; exact Shapley over all coalitions."""
    n = len(players)
    phi = {}
    for p in players:
        others = [q for q in players if q != p]
        tot = 0.0
        for r in range(len(others) + 1):
            for S in combinations(others, r):
                S = frozenset(S)
                w = math.factorial(len(S)) * math.factorial(n - len(S) - 1) / math.factorial(n)
                tot += w * (values[S | {p}] - values[S])
        phi[p] = tot
    return phi


# ══════════════════════════════════════════════════════════════════════════════
#  7. FIGURES FROM PREDICTION FILES  (comments 5, 6, 16, 17, 24, Q2a)
# ══════════════════════════════════════════════════════════════════════════════
def _ax_style(ax):
    ax.set_facecolor("#F6F8FA")
    for sp in ax.spines.values():
        sp.set_edgecolor("#D0D7DE")


def fig_confusion(cm, class_names, path, title="Confusion matrix"):
    cm = np.array(cm); cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cmn, cmap="Blues", vmin=0, vmax=1, annot=cm, fmt="d", cbar_kws={"label": "Row-normalised"},
                xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.6, linecolor="#D0D7DE")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_roc(y, prob, class_names, path, title="ROC"):
    lb = label_binarize(y, classes=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(7, 6)); _ax_style(ax)
    for k, c in enumerate(class_names):
        if lb[:, k].sum() in (0, len(y)):
            continue
        fpr, tpr, _ = roc_curve(lb[:, k], prob[:, k])
        ax.plot(fpr, tpr, color=PALETTE[k], lw=2, label=f"{c} (AUC={auc(fpr, tpr):.4f})")
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(title, fontweight="bold"); ax.legend(loc="lower right")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_per_class(metrics, path):
    names = list(metrics["per_class"].keys())
    x = np.arange(len(names)); w = 0.27
    fig, ax = plt.subplots(figsize=(8, 4.5)); _ax_style(ax)
    for i, key in enumerate(["precision", "recall", "f1"]):
        ax.bar(x + (i - 1) * w, [100 * metrics["per_class"][c][key] for c in names], w, label=key, color=PALETTE[i])
    ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(80, 101); ax.set_ylabel("Score (%)"); ax.legend(loc="lower right")
    ax.set_title("Per-class precision / recall / F1 (from saved predictions)", fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_reliability(prob_raw, prob_ts, y, path, n_bins=15, title="Reliability"):
    fig, ax = plt.subplots(figsize=(5.5, 5.5)); _ax_style(ax)
    ax.plot([0, 1], [0, 1], "--", color="grey")
    for pr, lab, col in [(prob_raw, "raw", PALETTE[0]), (prob_ts, "temp-scaled", PALETTE[1])]:
        if pr is None:
            continue
        x, a, _ = reliability_curve(pr, y, n_bins)
        ax.plot(x, a, "o-", color=col, label=f"{lab} (ECE={ece_score(pr, y, n_bins):.3f})")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.set_title(title, fontweight="bold"); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_risk_coverage(y, pred, prob, vac, path):
    correct = (y == pred)
    scores = {"1-maxprob": 1 - prob.max(1), "entropy": predictive_entropy(prob)}
    if np.isfinite(vac).any():
        scores = {"vacuity": vac, **scores}
    fig, ax = plt.subplots(figsize=(6, 4.5)); _ax_style(ax)
    for i, (n, s) in enumerate(scores.items()):
        cov, risk, aurc = risk_coverage(s, correct)
        ax.plot(cov, 100 * risk, color=PALETTE[i], label=f"{n} (AURC={100 * aurc:.2f}%)")
    ax.set_xlabel("Coverage"); ax.set_ylabel("Selective risk (%)"); ax.legend()
    ax.set_title("Risk–coverage", fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_uncertainty_hist(y, pred, vac, path):
    if not np.isfinite(vac).any():
        return
    correct = (y == pred)
    fig, ax = plt.subplots(figsize=(6, 4.5)); _ax_style(ax)
    bins = np.linspace(np.nanmin(vac), np.nanmax(vac), 40)
    ax.hist(vac[correct], bins=bins, alpha=0.7, color="#1A7F37", label=f"correct (n={correct.sum()})", density=True)
    if (~correct).any():
        ax.hist(vac[~correct], bins=bins, alpha=0.7, color="#D9480F", label=f"incorrect (n={(~correct).sum()})", density=True)
    ax.set_xlabel("Vacuity (K/S)"); ax.set_ylabel("Density"); ax.legend()
    ax.set_title("Vacuity: correct vs incorrect (density-normalised)", fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def fig_embedding(Z, y, class_names, path, title):
    fig, ax = plt.subplots(figsize=(7, 6)); _ax_style(ax)
    for k, c in enumerate(class_names):
        m = y == k
        ax.scatter(Z[m, 0], Z[m, 1], s=10, color=PALETTE[k], alpha=0.8, label=f"{c} (n={m.sum()})")
    ax.set_title(title, fontweight="bold"); ax.legend(); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
#  8. ONE-CALL EVALUATION OF A PREDICTION FILE
# ══════════════════════════════════════════════════════════════════════════════
def evaluate_predictions(df, out_prefix, class_names=CLASS_NAMES, val_df=None, subset_mask=None,
                         bootstrap=True, make_figs=True):
    """Everything the reviewer asked for, from one CSV. Returns a JSON-serialisable dict."""
    K = len(class_names)
    os.makedirs(os.path.dirname(out_prefix) or ".", exist_ok=True)
    if subset_mask is not None:
        df = df[subset_mask].reset_index(drop=True)
    y, pred, prob, vac = arrays_from_df(df, K)
    res = dict(metrics=classification_metrics(y, pred, prob, class_names))
    if bootstrap:
        res["bootstrap_ci95"] = bootstrap_ci(y, pred, prob)
    res["calibration_raw"] = calibration_report(prob, y)
    prob_ts = None
    if val_df is not None and len(val_df):
        yv, _, pv, _ = arrays_from_df(val_df, K)
        T = fit_temperature(pv, yv)                       # fitted on VALIDATION only
        prob_ts = apply_temperature(prob, T)
        res["temperature"] = T
        res["calibration_temp_scaled"] = calibration_report(prob_ts, y)
    res["error_detection"] = error_detection(y, pred, prob, vac)
    # per-class calibration / uncertainty (external no-tumor analysis, comment 25)
    res["per_class_uncertainty"] = {}
    for k, c in enumerate(class_names):
        m = y == k
        if m.any():
            res["per_class_uncertainty"][c] = dict(
                n=int(m.sum()), recall=float((pred[m] == k).mean()),
                mean_vacuity=float(np.nanmean(vac[m])) if np.isfinite(vac[m]).any() else None,
                mean_entropy=float(predictive_entropy(prob[m]).mean()),
                mean_maxprob=float(prob[m].max(1).mean()),
                ece=ece_score(prob[m], y[m]) if m.sum() >= 30 else None)
    with open(out_prefix + "_summary.json", "w") as f:
        json.dump(res, f, indent=2)
    high_confidence_errors(df, K).to_csv(out_prefix + "_high_conf_errors.csv", index=False)
    if make_figs:
        fig_confusion(res["metrics"]["confusion_matrix"], class_names, out_prefix + "_confusion.png")
        fig_roc(y, prob, class_names, out_prefix + "_roc.png")
        fig_per_class(res["metrics"], out_prefix + "_perclass.png")
        fig_reliability(prob, prob_ts, y, out_prefix + "_reliability.png")
        fig_risk_coverage(y, pred, prob, vac, out_prefix + "_risk_coverage.png")
        fig_uncertainty_hist(y, pred, vac, out_prefix + "_vacuity_hist.png")
    return res


def sanity_check_pred_df(df, K=4, tol=1e-3, name=""):
    """Fail loudly if probabilities / predictions are inconsistent (guards against comment-5 type errors)."""
    prob = df[[f"p{k}" for k in range(K)]].values.astype(np.float64)
    bad = ~np.isfinite(prob).all(1)
    if bad.any():
        raise AssertionError(f"[{name}] {int(bad.sum())}/{len(prob)} rows have NaN/inf probabilities -> the model produced "
                             f"non-finite outputs (fp16 overflow or diverged training). See NaN notes in the code.")
    sums = prob.sum(1)
    assert np.allclose(sums, 1.0, atol=tol), f"[{name}] probabilities do not sum to 1 (min={sums.min():.4f}, max={sums.max():.4f})"
    assert (prob.argmax(1) == df["pred"].values).all(), f"[{name}] pred column != argmax(prob)"
    return True


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  (everything the reviewer asked to be documented lives here -> run_config.json)
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    data_root="/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task",
    ext_data_root="/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset",
    ext_splits=["Training"],            # which Nickparvar folders form the external set (log the counts!)
    mask_root=None,                     # optional: folder with BRISC test masks for Grad-CAM hit-rate
    out_dir="/kaggle/working",
    img_size=224, num_classes=4, class_names=CLASS_NAMES,

    # training protocol (matches the notebook that produced the original numbers)
    epochs=50, batch_size=32, grad_accum=2, num_workers=2,
    lr=3e-4, weight_decay=1e-4, label_smooth=0.05, ema_decay=0.999, clip_grad=1.0,
    pct_start=0.15, div_factor=25.0, final_div_factor=1e4,
    backbone="efficientnet_b3", n_trainable_backbone_tensors=60,
    pretrained=True,                    # if True and weights cannot be loaded -> raise (no silent fallback)

    # architecture
    ssm_d_model=192, ssm_d_state=16, ssm_layers=3, ssm_patch=16,
    branch_dim=192, fusion_dim=384, n_tokens=49, attn_heads=4, dropout=0.25,

    # evidential loss
    lambda_kl=0.05, kl_anneal_epochs=10, max_evidence=100.0,

    # data hygiene
    val_frac=0.10, split_seed=42, dup_thr=0.985, drop_train_dups_of_test=True, drop_external_dups=True,

    # experiments
    seeds=[42, 43, 44, 45, 46],
    variants=["full", "softmax_head"],   # add: concat_mlp, mha_fusion, gated_fusion, cnn_only, cnn_b4, no_smooth, ...
    primary_variant="full",
    run_cv=True, cv_folds=5, cv_epochs=None,   # None -> same epochs as main run
    run_analysis=True, tsne_per_class=250, tta=True,
)

VARIANTS = {
    "full":         dict(fusion="tcafg",  head="evidential", branches=("wt", "ssm", "cnn")),
    "softmax_head": dict(fusion="tcafg",  head="softmax",    branches=("wt", "ssm", "cnn")),
    "no_smooth":    dict(fusion="tcafg",  head="evidential", branches=("wt", "ssm", "cnn"), label_smooth=0.0),
    "concat_mlp":   dict(fusion="concat", head="evidential", branches=("wt", "ssm", "cnn")),
    "mha_fusion":   dict(fusion="mha",    head="evidential", branches=("wt", "ssm", "cnn")),
    "gated_fusion": dict(fusion="gated",  head="evidential", branches=("wt", "ssm", "cnn")),
    "cnn_only":     dict(fusion="pool",   head="evidential", branches=("cnn",)),
    "cnn_b4":       dict(fusion="pool",   head="evidential", branches=("cnn",), backbone="efficientnet_b4"),
    "no_wavelet":   dict(fusion="tcafg",  head="evidential", branches=("ssm", "cnn")),
    "no_ssm":       dict(fusion="tcafg",  head="evidential", branches=("wt", "cnn")),
}

RUN_MODE = "smoke" if SMOKE else os.environ.get("RUN_MODE", "paper")      # "smoke" | "quick" | "paper"
if RUN_MODE == "quick":      # 2 variants x 3 seeds x 30 epochs, no CV
    CFG.update(seeds=[42, 43, 44], epochs=30, run_cv=False)
elif RUN_MODE == "smoke":    # tiny synthetic run to verify the whole pipeline end-to-end (CPU is fine)
    CFG.update(data_root="/tmp/smoke/brisc/classification_task", ext_data_root="/tmp/smoke/ext",
               out_dir="/tmp/smoke/out", epochs=2, cv_epochs=1, seeds=[42, 43], batch_size=4, grad_accum=1,
               num_workers=0, pretrained=False, variants=["full", "softmax_head"],
               tsne_per_class=8, cv_folds=2, n_trainable_backbone_tensors=10**6, img_size=112)
elif RUN_MODE != "paper":
    raise ValueError(f"RUN_MODE must be smoke | quick | paper, got {RUN_MODE!r}")
CFG.setdefault("time_budget_h", 11.5)         # everything (training + analysis) must end before this many hours after start
CFG.setdefault("analysis_reserve_min", 40)      # minutes kept free at the end for aggregation + analysis figures
CFG["resume"] = True                 # skip runs whose checkpoint + test predictions already exist in out_dir
T0 = float(os.environ.get("WAVESSM_T0", time.time()))            # wall-clock start, shared with the worker processes
print(f"[CONFIG] RUN_MODE={RUN_MODE}  seeds={CFG['seeds']}  epochs={CFG['epochs']}  variants={CFG['variants']}  cv={CFG['run_cv']}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda" and CFG.get("amp", True)
os.makedirs(CFG["out_dir"], exist_ok=True)
K = CFG["num_classes"]


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def seed_worker(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)


def vcfg(variant, key):
    """Variant-specific override of a CFG key."""
    return VARIANTS[variant].get(key, CFG[key])


# ══════════════════════════════════════════════════════════════════════════════
#  1. DATA: discovery, duplicate detection, group-aware splits
# ══════════════════════════════════════════════════════════════════════════════
NORM = {"notumor": "notumor", "no_tumor": "notumor", "no tumor": "notumor", "normal": "notumor",
        "glioma": "glioma", "meningioma": "meningioma", "pituitary": "pituitary"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def scan_split(split_dir):
    """Return DataFrame(path,label) for a folder with one sub-folder per class."""
    rows = []
    for d in sorted(os.listdir(split_dir)):
        full = os.path.join(split_dir, d)
        canon = NORM.get(d.lower())
        if not os.path.isdir(full) or canon not in CFG["class_names"]:
            continue
        for fn in sorted(os.listdir(full)):
            if os.path.splitext(fn)[1].lower() in IMG_EXT:
                rows.append((os.path.join(full, fn), CFG["class_names"].index(canon)))
    return pd.DataFrame(rows, columns=["path", "label"])


def find_dir(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None


def build_data():
    """Builds the master manifest. Roles: train / val / test / dropped_dup_of_test /
    external / external_dropped_dup."""
    root = CFG["data_root"]
    if not os.path.isdir(root):
        raise FileNotFoundError(f"BRISC root not found: {root}")
    tr_dir = find_dir(root, ["train", "Training", "Train", "TRAIN"])
    te_dir = find_dir(root, ["test", "Testing", "Test", "TEST"])
    if tr_dir is None or te_dir is None:
        raise FileNotFoundError(f"Could not find train/test folders under {root}")
    tr, te = scan_split(tr_dir), scan_split(te_dir)
    tr["origin"], te["origin"] = "brisc_train", "brisc_test"
    br = pd.concat([tr, te], ignore_index=True)
    print(f"[DATA] BRISC train={len(tr)}  test={len(te)}  class counts test={te.label.value_counts().sort_index().tolist()}")

    # ---- near-duplicate detection inside BRISC (comments 3, 4) -------------------
    cache = os.path.join(CFG["out_dir"], "brisc_thumbs.npy")
    T = thumbnail_matrix(br.path.tolist(), cache=cache)
    pairs = cosine_pairs(T, thr=CFG["dup_thr"])
    br["cluster"] = cluster_from_pairs(len(br), pairs)
    org = br.origin.values
    n_tt = sum((org[int(i)] == "brisc_train") != (org[int(j)] == "brisc_train") for i, j, _ in pairs)
    n_in_test = sum(org[int(i)] == "brisc_test" and org[int(j)] == "brisc_test" for i, j, _ in pairs)
    n_in_train = sum(org[int(i)] == "brisc_train" and org[int(j)] == "brisc_train" for i, j, _ in pairs)
    print(f"[DUP] near-duplicate pairs (cos>={CFG['dup_thr']}): train-train={n_in_train}  test-test={n_in_test}  train-test={n_tt}")
    dup_report = dict(train_train=int(n_in_train), test_test=int(n_in_test), train_test=int(n_tt), thr=CFG["dup_thr"])

    br["role"] = "train_pool"
    br.loc[br.origin == "brisc_test", "role"] = "test"
    test_clusters = set(br.loc[br.origin == "brisc_test", "cluster"])
    leak = (br.origin == "brisc_train") & br.cluster.isin(test_clusters)
    if CFG["drop_train_dups_of_test"]:
        br.loc[leak, "role"] = "dropped_dup_of_test"
    print(f"[DUP] train images sharing a cluster with a test image: {int(leak.sum())} "
          f"({'dropped' if CFG['drop_train_dups_of_test'] else 'kept'})")
    dup_report["train_dropped"] = int(leak.sum()) if CFG["drop_train_dups_of_test"] else 0

    # one representative per cluster inside the test set -> 'deduplicated test' metrics
    br["is_rep"] = False
    te_idx = br.index[br.role == "test"]
    first = br.loc[te_idx].groupby("cluster").head(1).index
    br.loc[first, "is_rep"] = True
    dup_report["test_unique"] = int(br.is_rep.sum())

    # ---- group-aware stratified train/val split (checkpoint-selection set) -------
    pool = br[br.role == "train_pool"]
    tr_i, va_i = group_stratified_split(pool.label.values, pool.cluster.values, CFG["val_frac"], CFG["split_seed"])
    br.loc[pool.index[va_i], "role"] = "val"
    br.loc[pool.index[tr_i], "role"] = "train"

    # ---- external cohort (comment 2) --------------------------------------------
    ext = None
    if CFG["ext_data_root"] and os.path.isdir(CFG["ext_data_root"]):
        parts = []
        for sp in CFG["ext_splits"]:
            d = find_dir(CFG["ext_data_root"], [sp, sp.lower(), sp.capitalize()])
            if d is not None:
                parts.append(scan_split(d).assign(origin=f"external_{sp}"))
        if parts:
            ext = pd.concat(parts, ignore_index=True)
            print(f"[EXT] {len(ext)} images  per-class counts (VERIFY against the paper's 5,600): "
                  f"{ext.label.value_counts().sort_index().tolist()}")
            Te = thumbnail_matrix(ext.path.tolist(), cache=os.path.join(CFG["out_dir"], "ext_thumbs.npy"))
            cross = cosine_pairs(Te, T, thr=CFG["dup_thr"])
            dup_ext = np.unique(cross[:, 0].astype(int)) if len(cross) else np.array([], int)
            ext["cluster"] = -1; ext["is_rep"] = True
            ext["role"] = "external"
            if CFG["drop_external_dups"] and len(dup_ext):
                ext.loc[dup_ext, "role"] = "external_dropped_dup"
            print(f"[EXT] images near-duplicating a BRISC image: {len(dup_ext)} "
                  f"({'dropped' if CFG['drop_external_dups'] else 'kept'})")
            dup_report["external_total"] = int(len(ext)); dup_report["external_dup_of_brisc"] = int(len(dup_ext))
    manifest = pd.concat([br, ext], ignore_index=True) if ext is not None else br
    manifest.to_csv(os.path.join(CFG["out_dir"], "split_manifest.csv"), index=False)
    json.dump(dup_report, open(os.path.join(CFG["out_dir"], "duplicate_report.json"), "w"), indent=2)
    summary = manifest.groupby(["role", "label"]).size().unstack(fill_value=0)
    summary.columns = [CFG["class_names"][c] for c in summary.columns]
    summary.to_csv(os.path.join(CFG["out_dir"], "dataset_distribution_table.csv"))
    print(summary)
    return manifest


def build_transforms(train):
    mean, std, sz = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], CFG["img_size"]
    if train:
        return transforms.Compose([
            transforms.Resize((sz + 32, sz + 32)), transforms.RandomCrop(sz), transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(12), transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), transforms.RandomGrayscale(p=0.05),
            transforms.ToTensor(), transforms.Normalize(mean, std), transforms.RandomErasing(p=0.2, scale=(0.02, 0.10))])
    return transforms.Compose([transforms.Resize((sz, sz)), transforms.ToTensor(), transforms.Normalize(mean, std)])


class ImgDataset(Dataset):
    def __init__(self, df, train):
        self.paths, self.labels = df.path.tolist(), df.label.tolist()
        self.tf = build_transforms(train)

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.tf(im.convert("RGB"))
        return x, self.labels[i], i


def make_loader(df, train, seed=0, bs=None):
    ds = ImgDataset(df, train)
    bs = bs or CFG["batch_size"]
    g = torch.Generator(); g.manual_seed(seed)
    if train:
        cnt = np.bincount(ds.labels, minlength=K).clip(min=1)
        w = (1.0 / cnt)[np.array(ds.labels)]
        sampler = WeightedRandomSampler(torch.from_numpy(w).float(), len(w), replacement=True, generator=g)
        return DataLoader(ds, bs, sampler=sampler, num_workers=CFG["num_workers"], pin_memory=USE_AMP,
                          worker_init_fn=seed_worker, generator=g, drop_last=len(ds) > bs)
    return DataLoader(ds, bs, shuffle=False, num_workers=CFG["num_workers"], pin_memory=USE_AMP, worker_init_fn=seed_worker)


# ══════════════════════════════════════════════════════════════════════════════
#  2. MODEL
# ══════════════════════════════════════════════════════════════════════════════
class HaarDWT(nn.Module):
    """Single-level separable 2-D Haar DWT as fixed stride-2 depthwise 2x2 convolutions.
    LL = h(x)h, LH = g(rows)h(cols) [horizontal detail], HL = h(rows)g(cols) [vertical detail], HH = g g,
    with h=[1,1]/sqrt2, g=[1,-1]/sqrt2.  (12 x 112 x 112 for a 3 x 224 x 224 input; no padding: even size.)"""
    def __init__(self, c=3):
        super().__init__()
        h = torch.tensor([1.0, 1.0]) / math.sqrt(2); g = torch.tensor([1.0, -1.0]) / math.sqrt(2)
        k = torch.stack([torch.outer(h, h), torch.outer(g, h), torch.outer(h, g), torch.outer(g, g)]).unsqueeze(1)
        self.register_buffer("weight", k.repeat(c, 1, 1, 1)); self.c = c

    def forward(self, x):
        return F.conv2d(x.float(), self.weight, stride=2, groups=self.c).to(x.dtype)


def test_haar_vs_pywt():
    """Unit test (comment 8): LL identical; detail bands equal up to sign to pywt 'haar'."""
    try:
        import pywt
    except ImportError:
        print("[TEST] pywt not installed -> skipped"); return
    x = torch.randn(1, 1, 16, 16); m = HaarDWT(1)
    out = m(x)[0].numpy(); cA, (cH, cV, cD) = pywt.dwt2(x[0, 0].numpy(), "haar")
    ok = [np.allclose(out[0], cA, atol=1e-5)] + [np.allclose(np.abs(out[i + 1]), np.abs(b), atol=1e-5)
                                                  for i, b in enumerate([cH, cV, cD])]
    print(f"[TEST] Haar DWT vs PyWavelets (LL, LH, HL, HH): {ok}")
    assert all(ok), "DWT mismatch"


class WaveletBranch(nn.Module):
    def __init__(self, out_dim=192, c=3):
        super().__init__()
        self.dwt = HaarDWT(c); ch = 4 * c
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch, 2 * ch), nn.ReLU(True),
                                  nn.Linear(2 * ch, ch), nn.Sigmoid())
        self.encoder = nn.Sequential(
            nn.Conv2d(ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, padding=1, stride=2, bias=False), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 256, 3, padding=1, stride=2, bias=False), nn.BatchNorm2d(256), nn.GELU())
        self.proj = nn.Linear(256, out_dim); self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        wt = self.dwt(x); wt = wt * self.gate(wt)[:, :, None, None]
        fmap = self.encoder(wt)                                        # (B,256,28,28)
        pooled = F.adaptive_avg_pool2d(fmap, 7).flatten(2).transpose(1, 2)   # (B,49,256)
        return self.norm(self.proj(pooled)), fmap


def parallel_scan(dA, dBx):
    """Inclusive scan of h_t = dA_t * h_{t-1} + dBx_t (Hillis-Steele, O(L log L))."""
    B, L, D, S = dA.shape
    a, b, stride = dA, dBx, 1
    while stride < L:
        na = a[:, stride:] * a[:, :-stride]
        nb = a[:, stride:] * b[:, :-stride] + b[:, stride:]
        a = torch.cat([a[:, :stride], na], 1); b = torch.cat([b[:, :stride], nb], 1); stride *= 2
    return b


class SelectiveSSM(nn.Module):
    """Diagonal selective SSM. Discretisation actually used (matches Eqs. 2-4 in the revised paper):
       A = -exp(A_log) (<0), Abar = exp(Delta*A) elementwise, Bbar ~= mean_d(Delta) * B  (first-order)."""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model)
        self.conv1d = nn.Conv1d(d_model, d_model, 3, padding=1, groups=d_model)
        self.x_proj = nn.Linear(d_model, 2 * d_state + 1)
        self.dt_proj = nn.Linear(1, d_model)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)).unsqueeze(0).expand(d_model, -1).clone())
        self.D = nn.Parameter(torch.ones(d_model))
        self.out_proj = nn.Linear(d_model, d_model); self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        with torch.autocast("cuda", enabled=False):                    # whole SSM in fp32: the scan must not overflow in fp16
            return self._forward(x.float())

    def _forward(self, x):
        S, res = self.d_state, x
        x_, z = self.in_proj(x).chunk(2, -1)
        x_c = F.silu(self.conv1d(x_.transpose(1, 2)).transpose(1, 2))
        p = self.x_proj(x_c)
        B_, C_ = p[..., :S], p[..., S:2 * S]
        delta = F.softplus(self.dt_proj(p[..., -1:])) + 1e-4
        A = -torch.exp(self.A_log.float())
        dA = torch.exp(delta.float().unsqueeze(-1) * A[None, None])
        dB = delta.mean(-1, keepdim=True).float() * B_.float()
        dBx = x_c.float().unsqueeze(-1) * dB.unsqueeze(2)
        h = parallel_scan(dA, dBx)
        y = (h * C_.float().unsqueeze(2)).sum(-1).to(x_c.dtype) + self.D * x_c
        return self.norm(self.out_proj(y * F.silu(z)) + res)


class SSMBlock(nn.Module):
    def __init__(self, d, s):
        super().__init__()
        self.n1 = nn.LayerNorm(d); self.ssm = SelectiveSSM(d, s); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(0.1), nn.Linear(4 * d, d))

    def forward(self, x):
        x = self.ssm(self.n1(x)) + x
        return self.ff(self.n2(x)) + x


class SSMBranch(nn.Module):
    """Operates on RAW RGB 16x16 patches (196 tokens), unidirectional raster scan."""
    def __init__(self, img=224, patch=16, d=192, s=16, layers=3, out_dim=192):
        super().__init__()
        self.g = img // patch
        self.unfold = nn.Unfold(patch, stride=patch)
        self.patch_proj = nn.Linear(3 * patch * patch, d)
        self.pos = nn.Parameter(torch.randn(1, self.g ** 2, d) * 0.02)
        self.blocks = nn.ModuleList([SSMBlock(d, s) for _ in range(layers)])
        self.norm = nn.LayerNorm(d); self.proj = nn.Linear(d, out_dim); self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        t = self.patch_proj(self.unfold(x).transpose(1, 2)) + self.pos
        for b in self.blocks:
            t = checkpoint(b, t, use_reentrant=False) if (self.training and torch.is_grad_enabled()) else b(t)   # saves ~3 GB
        t = self.norm(t)                                               # (B,196,d)
        fmap = t.transpose(1, 2).reshape(t.size(0), -1, self.g, self.g)
        pooled = F.adaptive_avg_pool2d(fmap, 7).flatten(2).transpose(1, 2)   # (B,49,d)
        return self.out_norm(self.proj(pooled)), fmap


class SpatialBranch(nn.Module):
    def __init__(self, backbone="efficientnet_b3", out_dim=192, dropout=0.25):
        super().__init__()
        try:
            self.backbone = timm.create_model(backbone, pretrained=CFG["pretrained"], num_classes=0, global_pool="")
        except Exception as e:
            raise RuntimeError(f"Could not create {backbone} (pretrained={CFG['pretrained']}): {e}. "
                               "Enable Internet / attach the weights; silent fallback is disabled (comment 27).")
        self.pretrained_loaded = bool(CFG["pretrained"])
        params = list(self.backbone.parameters())
        for p in params[:-CFG["n_trainable_backbone_tensors"]] if CFG["n_trainable_backbone_tensors"] < len(params) else []:
            p.requires_grad_(False)
        self.proj = nn.Sequential(nn.Linear(self.backbone.num_features, out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(dropout))

    def forward(self, x):
        fmap = self.backbone.forward_features(x)                       # (B,C,7,7)
        t = F.adaptive_avg_pool2d(fmap, 7).flatten(2).transpose(1, 2)  # (B,49,C)
        return self.proj(t), fmap


# ---- fusion modules (all take dict name->(B,N,D) tokens, return (B,fusion_dim)) --------------
class DirectionalCrossAttn(nn.Module):
    """Tokens of branch i (queries) attend to tokens of branch j (keys/values); gated, then pooled."""
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.nq, self.nkv = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout * 0.4, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(2 * dim, dim), nn.Sigmoid())

    def forward(self, q, kv, need_weights=False):
        kvn = self.nkv(kv)
        out, w = self.attn(self.nq(q), kvn, kvn, need_weights=need_weights, average_attn_weights=True)
        y = q + out                                                    # (B,Nq,D)
        a = y.mean(1)
        g = self.gate(torch.cat([q.mean(1), a], -1))                   # pair-specific gate
        return g * a, w


class TCAFG(nn.Module):
    """Triple Cross-Attention Fusion Gate over token sequences. For branches (w,s,c): six directed
    attentions (w<-s, w<-c, s<-w, s<-c, c<-w, c<-s); each output has its own sigmoid gate."""
    def __init__(self, names, dim=192, out_dim=384, heads=4, n_tokens=49, dropout=0.25):
        super().__init__()
        self.names = list(names)
        self.pairs = [(a, b) for a in names for b in names if a != b]
        self.pos = nn.ParameterDict({n: nn.Parameter(torch.randn(1, n_tokens, dim) * 0.02) for n in names})
        self.ca = nn.ModuleDict({f"{a}_{b}": DirectionalCrossAttn(dim, heads, dropout) for a, b in self.pairs})
        f = dim * len(self.pairs)
        self.mlp = nn.Sequential(nn.LayerNorm(f), nn.Linear(f, 2 * out_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(2 * out_dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        t = {n: toks[n] + self.pos[n] for n in self.names}
        outs, attn = [], {}
        for a, b in self.pairs:
            o, w = self.ca[f"{a}_{b}"](t[a], t[b], need_weights=return_attn)
            outs.append(o)
            if return_attn: attn[f"{a}<-{b}"] = w
        z = self.mlp(torch.cat(outs, -1))
        return (z, attn) if return_attn else z


class ConcatMLPFusion(nn.Module):
    def __init__(self, names, dim=192, out_dim=384, dropout=0.25, **_):
        super().__init__(); self.names = list(names)
        f = dim * len(names)
        self.mlp = nn.Sequential(nn.LayerNorm(f), nn.Linear(f, 2 * out_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(2 * out_dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        z = self.mlp(torch.cat([toks[n].mean(1) for n in self.names], -1))
        return (z, {}) if return_attn else z


class MHAFusion(nn.Module):
    """Standard multi-head self-attention fusion (CLS + all tokens, 2 transformer layers)."""
    def __init__(self, names, dim=192, out_dim=384, heads=4, n_tokens=49, dropout=0.25, **_):
        super().__init__(); self.names = list(names)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.type_emb = nn.ParameterDict({n: nn.Parameter(torch.randn(1, 1, dim) * 0.02) for n in names})
        layer = nn.TransformerEncoderLayer(dim, heads, dim_feedforward=2 * dim, dropout=dropout * 0.4,
                                           batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, 2)
        self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        seq = torch.cat([self.cls.expand(toks[self.names[0]].size(0), -1, -1)] +
                        [toks[n] + self.type_emb[n] for n in self.names], 1)
        z = self.out(self.enc(seq)[:, 0])
        return (z, {}) if return_attn else z


class GatedFusion(nn.Module):
    def __init__(self, names, dim=192, out_dim=384, dropout=0.25, **_):
        super().__init__(); self.names = list(names)
        self.gate = nn.Linear(dim * len(names), dim * len(names))
        f = dim * len(names)
        self.mlp = nn.Sequential(nn.LayerNorm(f), nn.Linear(f, 2 * out_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(2 * out_dim, out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        v = torch.cat([toks[n].mean(1) for n in self.names], -1)
        z = self.mlp(v * torch.sigmoid(self.gate(v)))
        return (z, {}) if return_attn else z


class PoolFusion(nn.Module):
    """Single-branch baseline (EfficientNet-only): mean-pool tokens, project to fusion_dim."""
    def __init__(self, names, dim=192, out_dim=384, **_):
        super().__init__(); self.names = list(names)
        self.out = nn.Sequential(nn.Linear(dim * len(names), out_dim), nn.LayerNorm(out_dim))

    def forward(self, toks, return_attn=False):
        z = self.out(torch.cat([toks[n].mean(1) for n in self.names], -1))
        return (z, {}) if return_attn else z


FUSIONS = dict(tcafg=TCAFG, concat=ConcatMLPFusion, mha=MHAFusion, gated=GatedFusion, pool=PoolFusion)


class Head(nn.Module):
    """3-layer MLP 384->512->256->K (329,476 params for K=4). kind: evidential | softmax."""
    def __init__(self, in_dim, kind, dropout=0.25):
        super().__init__(); self.kind = kind
        self.mlp = nn.Sequential(nn.Linear(in_dim, 512), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(512, 256), nn.GELU(), nn.Dropout(dropout / 2), nn.Linear(256, K))

    def forward(self, x):
        logits = self.mlp(x).float()
        if self.kind == "evidential":
            ev = F.softplus(logits.clamp(-20, 20)).clamp(max=CFG["max_evidence"])
            alpha = ev + 1.0
            S = alpha.sum(1, keepdim=True)
            return dict(prob=alpha / S, alpha=alpha, vac=(K / S).squeeze(1), logits=logits)
        prob = F.softmax(logits, 1)
        ent = -(prob * torch.log(prob.clamp_min(1e-12))).sum(1) / math.log(K)
        return dict(prob=prob, alpha=None, vac=ent * float("nan"), logits=logits)   # vacuity undefined


class WaveSSMTriNet(nn.Module):
    def __init__(self, variant="full"):
        super().__init__()
        v = VARIANTS[variant]; B = CFG["branch_dim"]
        self.variant, self.branches = variant, tuple(v["branches"])
        if "wt" in self.branches: self.wavelet_branch = WaveletBranch(B)
        if "ssm" in self.branches:
            self.ssm_branch = SSMBranch(CFG["img_size"], CFG["ssm_patch"], CFG["ssm_d_model"], CFG["ssm_d_state"], CFG["ssm_layers"], B)
        if "cnn" in self.branches:
            self.cnn_branch = SpatialBranch(v.get("backbone", CFG["backbone"]), B, CFG["dropout"])
        self.fusion = FUSIONS[v["fusion"]](self.branches, dim=B, out_dim=CFG["fusion_dim"], heads=CFG["attn_heads"],
                                           n_tokens=CFG["n_tokens"], dropout=CFG["dropout"])
        self.head = Head(CFG["fusion_dim"], v["head"], CFG["dropout"])
        for n in self.branches:                                         # training-mean tokens for occlusion (#22)
            self.register_buffer(f"mean_{n}", torch.zeros(1, CFG["n_tokens"], B))
        self.capture, self._maps = False, {}

    def branch_tokens(self, x):
        toks, maps = {}, {}
        for n, mod in (("wt", "wavelet_branch"), ("ssm", "ssm_branch"), ("cnn", "cnn_branch")):
            if n in self.branches:
                toks[n], maps[n] = getattr(self, mod)(x)
        return toks, maps

    def forward(self, x, drop=(), return_attn=False):
        toks, maps = self.branch_tokens(x)
        for n in drop:                                                   # occlusion: replace by training-mean tokens
            if n in toks: toks[n] = getattr(self, f"mean_{n}").expand_as(toks[n]).to(toks[n].dtype)
        if self.capture:
            for m in maps.values():
                if m.requires_grad: m.retain_grad()
            self._maps = maps
        fz = self.fusion(toks, return_attn=return_attn)
        fused, attn = fz if return_attn else (fz, None)
        out = self.head(fused); out["fused"] = fused
        if return_attn: out["attn"] = attn
        return out


# ══════════════════════════════════════════════════════════════════════════════
#  3. LOSSES  (comment 15)
# ══════════════════════════════════════════════════════════════════════════════
def kl_dirichlet_to_uniform(alpha):
    K_ = alpha.shape[1]; a0 = alpha.sum(1)
    return (torch.lgamma(a0) - torch.lgamma(alpha).sum(1) - math.lgamma(K_)
            + ((alpha - 1) * (torch.digamma(alpha) - torch.digamma(a0.unsqueeze(1)))).sum(1)).mean()


class EvidentialLoss(nn.Module):
    """Expected squared error (Brier-type) + KL(Dir(alpha_tilde) || Dir(1)); NOT expected cross-entropy.
    alpha_tilde removes target-class evidence using the HARD one-hot label (smoothed label is used only in the MSE term)."""
    def __init__(self, smooth):
        super().__init__(); self.smooth = smooth

    def forward(self, out, y, epoch):
        with torch.autocast("cuda", enabled=False):
            alpha = out["alpha"].float()
            yh = F.one_hot(y, K).float(); ys = yh * (1 - self.smooth) + self.smooth / K
            S = alpha.sum(1, keepdim=True); p = alpha / S
            mse = ((ys - p) ** 2 + p * (1 - p) / (S + 1)).sum(1).mean()
            lam = CFG["lambda_kl"] * min(1.0, epoch / CFG["kl_anneal_epochs"])
            return mse + lam * kl_dirichlet_to_uniform(yh + (1 - yh) * alpha)


class SoftmaxLoss(nn.Module):
    def __init__(self, smooth):
        super().__init__(); self.smooth = smooth

    def forward(self, out, y, epoch):
        return F.cross_entropy(out["logits"].float(), y, label_smoothing=self.smooth)


def make_criterion(variant):
    s = vcfg(variant, "label_smooth")
    return (EvidentialLoss if VARIANTS[variant]["head"] == "evidential" else SoftmaxLoss)(s)


# ══════════════════════════════════════════════════════════════════════════════
#  4. TRAIN / PREDICT
# ══════════════════════════════════════════════════════════════════════════════
class ModelEMA:
    def __init__(self, model, decay):
        self.decay, self.shadow = decay, copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for se, e in zip(self.shadow.state_dict().values(), model.state_dict().values()):
            se.mul_(self.decay).add_(e.detach(), alpha=1 - self.decay) if se.dtype.is_floating_point else se.copy_(e)


@torch.no_grad()
def predict(model, loader, tta=None, with_fused=False):
    """Returns dict of arrays. TTA = horizontal flip: alpha (evidential) or probabilities (softmax) averaged."""
    tta = CFG["tta"] if tta is None else tta
    model.eval(); R = {k: [] for k in ["prob", "alpha", "vac", "label", "idx", "fused"]}
    for x, y, i in loader:
        x = x.to(DEVICE)
        with torch.autocast("cuda", enabled=False):                      # fp32 inference: no fp16 overflow -> NaN
            o = model(x.float()); o2 = model(torch.flip(x, [3]).float()) if tta else None
        if o["alpha"] is not None:
            alpha = o["alpha"] if o2 is None else (o["alpha"] + o2["alpha"]) / 2
            S = alpha.sum(1, keepdim=True); prob, vac = alpha / S, (K / S).squeeze(1)
        else:
            prob = o["prob"] if o2 is None else (o["prob"] + o2["prob"]) / 2
            alpha, vac = None, torch.full((x.size(0),), float("nan"), device=x.device)
        R["prob"].append(prob.float().cpu()); R["vac"].append(vac.float().cpu())
        R["label"].append(y); R["idx"].append(i)
        if alpha is not None: R["alpha"].append(alpha.float().cpu())
        if with_fused: R["fused"].append(o["fused"].float().cpu())
    out = {k: torch.cat(v).numpy() for k, v in R.items() if len(v)}
    return out


def preds_to_df(P, meta):
    """meta: DataFrame aligned with loader order. Saves the canonical prediction schema."""
    m = meta.iloc[P["idx"]].reset_index(drop=True)
    df = pd.DataFrame(P["prob"], columns=[f"p{k}" for k in range(K)])
    df.insert(0, "path", m.path.values); df.insert(1, "label", P["label"].astype(int))
    df.insert(2, "pred", P["prob"].argmax(1))
    if "alpha" in P:
        for k in range(K): df[f"alpha{k}"] = P["alpha"][:, k]
    df["vacuity"] = P["vac"]
    for c in ["cluster", "is_rep"]:
        if c in m: df[c] = m[c].values
    return df


def run_epoch(model, loader, criterion, opt, scaler, sched, epoch, ema, lr_log):
    model.train(); opt.zero_grad(set_to_none=True)
    tl = tc = tn = 0
    for step, (x, y, _) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(x)
        loss = criterion(out, y, epoch) / CFG["grad_accum"]
        if not torch.isfinite(loss):                                     # NaN/inf guard: skip this batch, never backprop it
            nbad = globals().get("_NAN_BATCHES", 0) + 1; globals()["_NAN_BATCHES"] = nbad
            print(f"  [WARN] non-finite loss at step {step} (total {nbad}); batch skipped")
            if nbad > 50: raise RuntimeError("Too many non-finite losses: training diverged. Lower lr (e.g. 1e-4) or set amp off.")
            opt.zero_grad(set_to_none=True); continue
        scaler.scale(loss).backward()
        if (step + 1) % CFG["grad_accum"] == 0 or step + 1 == len(loader):
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_grad"])
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
            ema.update(model); lr_log.append(opt.param_groups[0]["lr"])
        tl += loss.item() * CFG["grad_accum"] * y.size(0); tc += (out["prob"].argmax(1) == y).sum().item(); tn += y.size(0)
    return tl / tn, tc / tn


@torch.no_grad()
def eval_val(model, loader, criterion, epoch):
    """Validation loss/accuracy/NLL on the HELD-OUT validation split (never the test split)."""
    model.eval(); tl = tc = tn = nll = 0
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", enabled=False):
            out = model(x)
        tl += criterion(out, y, epoch).item() * y.size(0)
        tc += (out["prob"].argmax(1) == y).sum().item(); tn += y.size(0)
        nll += -torch.log(out["prob"].gather(1, y[:, None]).clamp_min(1e-12)).sum().item()
    return tl / tn, tc / tn, nll / tn


class BudgetExceeded(Exception):
    pass


TRAIN_DEADLINE = None        # absolute time (s) after which no training may be running; set by worker_main()


def train_model(variant, seed, train_df, val_df, epochs, tag):
    set_seed(seed)
    model = WaveSSMTriNet(variant).to(DEVICE)
    crit = make_criterion(variant)
    tr_loader, va_loader = make_loader(train_df, True, seed), make_loader(val_df, False)
    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = math.ceil(len(tr_loader) / CFG["grad_accum"]) * epochs + 1
    sched = optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps, pct_start=CFG["pct_start"],
                                          anneal_strategy="cos", div_factor=CFG["div_factor"], final_div_factor=CFG["final_div_factor"])
    scaler, ema = torch.amp.GradScaler("cuda", enabled=USE_AMP), ModelEMA(model, CFG["ema_decay"])
    hist, lr_log, ep_times, best = [], [], [], dict(key=(-1, 0), state=None, src="", ep=0)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, tr_loader, crit, opt, scaler, sched, ep, ema, lr_log)
        (vl, va, vn), (_, ea, en) = eval_val(model, va_loader, crit, ep), eval_val(ema.shadow, va_loader, crit, ep)
        for src, acc, nll, net in (("raw", va, vn, model), ("ema", ea, en, ema.shadow)):
            key = (acc, -nll)                                            # select on VAL accuracy, tie-break VAL NLL
            if not math.isfinite(nll):
                print(f"  [WARN] {src} val NLL is non-finite at epoch {ep}; not eligible for selection"); continue
            if key > best["key"]:
                best.update(key=key, state=copy.deepcopy(net.state_dict()), src=src, ep=ep)
        hist.append(dict(epoch=ep, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va, ema_val_acc=ea, lr=opt.param_groups[0]["lr"]))
        print(f"  [{tag}] ep {ep:3d} tr {tl:.4f}/{ta*100:5.2f}  val {vl:.4f}/{va*100:5.2f}  ema {ea*100:5.2f}  {time.time()-t0:.0f}s", flush=True)
        ep_times.append(time.time() - t0)
        if ep == 1: print(f"  [{tag}] projected run time ~ {ep_times[0]*epochs/60:.0f} min for {epochs} epochs", flush=True)
        if TRAIN_DEADLINE and ep < epochs and time.time() + np.mean(ep_times[-3:]) * (epochs - ep) > TRAIN_DEADLINE + 600:
            raise BudgetExceeded(f"{tag}: projected end is past the time budget -> run aborted at epoch {ep}/{epochs}")
    if best["state"] is None: raise RuntimeError("No finite checkpoint was produced (all epochs gave NaN/inf). Try USE_AMP=False / lower lr.")
    print(f"  [{tag}] selected epoch {best['ep']} ({best['src']}) val_acc={best['key'][0]*100:.2f}")
    model.load_state_dict(best["state"])
    ck = os.path.join(CFG["out_dir"], f"ckpt_{tag}.pt"); torch.save(best["state"], ck)
    pd.DataFrame(hist).to_csv(os.path.join(CFG["out_dir"], f"history_{tag}.csv"), index=False)
    np.save(os.path.join(CFG["out_dir"], f"lr_steps_{tag}.npy"), np.array(lr_log))
    return model, dict(selected_epoch=best["ep"], selected_src=best["src"], val_acc=best["key"][0],
                       pretrained=getattr(getattr(model, "cnn_branch", None), "pretrained_loaded", None))


def eval_all_sets(model, manifest, tag, variant):
    """Scores val / test / external ONCE with the selected checkpoint, saves prediction CSVs."""
    res = {}
    out = CFG["out_dir"]
    for name, role in (("val", "val"), ("test", "test"), ("external", "external")):
        meta = manifest[manifest.role == role].reset_index(drop=True)
        if not len(meta): continue
        P = predict(model, make_loader(meta, False))
        df = preds_to_df(P, meta); sanity_check_pred_df(df, K, name=f"{tag}/{name}")
        df.to_csv(os.path.join(out, f"pred_{tag}_{name}.csv"), index=False)
        res[name] = df
    return res


# ══════════════════════════════════════════════════════════════════════════════
#  5. EXPERIMENT DRIVER + CROSS-SEED STATISTICS
# ══════════════════════════════════════════════════════════════════════════════
def main_job(manifest, variant, seed):
    """Train (or resume) one (variant, seed), score val/test/external once, write predictions + metric JSONs."""
    tr, va = manifest[manifest.role == "train"], manifest[manifest.role == "val"]
    tag = f"{variant}_s{seed}"
    print(f"\n{'='*70}\n  {tag}\n{'='*70}", flush=True)
    ck = os.path.join(CFG["out_dir"], f"ckpt_{tag}.pt")
    resumed = False
    if CFG["resume"] and os.path.exists(ck) and os.path.exists(os.path.join(CFG["out_dir"], f"pred_{tag}_test.csv")):
        print(f"  [RESUME] {tag}: checkpoint found, skipping training", flush=True)
        model = WaveSSMTriNet(variant).to(DEVICE); model.load_state_dict(torch.load(ck, map_location=DEVICE)); model.eval(); resumed = True
    else:
        model, _ = train_model(variant, seed, tr, va, CFG["epochs"], tag)
    dfs = eval_all_sets(model, manifest, tag, variant)
    for name in ("test", "external"):
        if name not in dfs: continue
        d = dfs[name]; first = (seed == CFG["seeds"][0])
        evaluate_predictions(d, os.path.join(CFG["out_dir"], f"eval_{tag}_{name}"), val_df=dfs["val"], bootstrap=first, make_figs=first)
        if name == "test" and "is_rep" in d:
            evaluate_predictions(d, os.path.join(CFG["out_dir"], f"eval_{tag}_test_dedup"), val_df=dfs["val"],
                                 subset_mask=d.is_rep.values.astype(bool), bootstrap=False, make_figs=False)
    del model; torch.cuda.empty_cache()
    return resumed


def load_runs():
    """Rebuild the per-run summaries from files on disk (workers are separate processes)."""
    runs, out = {}, CFG["out_dir"]
    for variant in CFG["variants"]:
        for seed in CFG["seeds"]:
            tag = f"{variant}_s{seed}"
            summ, dfs = {}, {}
            for name in ("test", "test_dedup", "external"):
                p = os.path.join(out, f"eval_{tag}_{name}_summary.json")
                if os.path.exists(p): summ[name] = json.load(open(p))
            for name in ("val", "test", "external"):
                p = os.path.join(out, f"pred_{tag}_{name}.csv")
                if os.path.exists(p): dfs[name] = pd.read_csv(p)
            if "test" in summ and "test" in dfs:
                runs[(variant, seed)] = dict(summary=summ, dfs=dfs)
    missing = [f"{v}_s{s}" for v in CFG["variants"] for s in CFG["seeds"] if (v, s) not in runs]
    print(f"[RUNS] completed {len(runs)}/{len(CFG['variants']) * len(CFG['seeds'])}" + (f"   NOT completed: {missing}" if missing else ""))
    return runs


def aggregate(runs):
    rows, tests = [], []
    metrics = ["accuracy", "macro_f1", "mcc", "kappa", "mean_auc"]
    seeds_of = {v: [s for s in CFG["seeds"] if (v, s) in runs] for v in CFG["variants"]}
    for variant in CFG["variants"]:
        for setname in ("test", "test_dedup", "external"):
            have = [s for s in seeds_of[variant] if setname in runs[(variant, s)]["summary"]]
            if not have: continue
            vals = {m: [runs[(variant, s)]["summary"][setname]["metrics"][m] for s in have] for m in metrics}
            row = dict(variant=variant, set=setname, n_seeds=len(have), seeds=",".join(map(str, have)))
            for m in metrics:
                st = summarize_seeds(vals[m]); row[f"{m}_mean"], row[f"{m}_sd"] = st["mean"], st["sd"]
                row[f"{m}_ci95_lo"], row[f"{m}_ci95_hi"] = st["ci95"]
            for cal in ("ece", "brier", "nll"):
                c = [runs[(variant, s)]["summary"][setname]["calibration_raw"][cal] for s in have]
                row[f"{cal}_mean"] = float(np.mean(c)); row[f"{cal}_sd"] = float(np.std(c, ddof=1)) if len(c) > 1 else float("nan")
            rows.append(row)
    if not rows:
        print("[AGG] no completed runs -> nothing to aggregate"); return
    pd.DataFrame(rows).to_csv(os.path.join(CFG["out_dir"], "table_multiseed_summary.csv"), index=False)
    # paired comparisons vs the primary variant, on the seeds both variants completed
    base = CFG["primary_variant"]
    for variant in CFG["variants"]:
        if variant == base: continue
        common = [s for s in CFG["seeds"] if (base, s) in runs and (variant, s) in runs]
        for setname in ("test", "external"):
            ok = [s for s in common if setname in runs[(base, s)]["dfs"] and setname in runs[(variant, s)]["dfs"]]
            if len(ok) < 2: continue
            a = [runs[(base, s)]["summary"][setname]["metrics"]["accuracy"] for s in ok]
            b = [runs[(variant, s)]["summary"][setname]["metrics"]["accuracy"] for s in ok]
            t = paired_seed_tests(a, b); t.update(variant=variant, vs=base, set=setname, seeds=ok)
            t["mcnemar_p_per_seed"] = [mcnemar_exact((runs[(base, s)]["dfs"][setname].pred == runs[(base, s)]["dfs"][setname].label).values,
                                                        (runs[(variant, s)]["dfs"][setname].pred == runs[(variant, s)]["dfs"][setname].label).values)["p"]
                                       for s in ok]
            tests.append(t)
    json.dump(tests, open(os.path.join(CFG["out_dir"], "table_paired_tests.json"), "w"), indent=2)
    print(pd.DataFrame(rows)[["variant", "set", "n_seeds", "accuracy_mean", "accuracy_sd", "macro_f1_mean", "mean_auc_mean", "ece_mean"]])


# ══════════════════════════════════════════════════════════════════════════════
#  6. CROSS-VALIDATION  (group-aware, inner validation for checkpoint selection; comment 7)
# ══════════════════════════════════════════════════════════════════════════════
def cv_split(manifest, f):
    pool = manifest[manifest.origin.isin(["brisc_train", "brisc_test"])].reset_index(drop=True)
    skf = StratifiedGroupKFold(CFG["cv_folds"], shuffle=True, random_state=CFG["split_seed"])
    tr_i, te_i = list(skf.split(np.zeros(len(pool)), pool.label.values, pool.cluster.values))[f - 1]
    return pool, tr_i, te_i


def cv_job(manifest, f):
    """One group-aware CV fold with an inner validation split for checkpoint selection (never the held-out fold)."""
    pool, tr_i, te_i = cv_split(manifest, f)
    ep = CFG["cv_epochs"] or CFG["epochs"]
    tag = f"cv{f}"
    print(f"\n{'='*70}\n  {tag}\n{'='*70}", flush=True)
    pf = os.path.join(CFG["out_dir"], f"pred_{tag}_test.csv")
    if CFG["resume"] and os.path.exists(pf):
        print(f"  [RESUME] {tag}: predictions found, skipping training", flush=True); return True
    tr_all = pool.iloc[tr_i].reset_index(drop=True)
    a, b = group_stratified_split(tr_all.label.values, tr_all.cluster.values, CFG["val_frac"], CFG["split_seed"] + f)
    model, _ = train_model(CFG["primary_variant"], CFG["seeds"][0], tr_all.iloc[a], tr_all.iloc[b], ep, tag)
    te = pool.iloc[te_i].reset_index(drop=True)
    df = preds_to_df(predict(model, make_loader(te, False)), te); sanity_check_pred_df(df, K, name=tag)
    df.to_csv(pf, index=False); del model; torch.cuda.empty_cache()
    return False


def cv_aggregate():
    rows = []
    for f in range(1, CFG["cv_folds"] + 1):
        pf = os.path.join(CFG["out_dir"], f"pred_cv{f}_test.csv")
        if not os.path.exists(pf): continue
        df = pd.read_csv(pf)
        m = classification_metrics(df.label.values, df.pred.values, df[[f"p{k}" for k in range(K)]].values)
        rows.append(dict(fold=f, **{k: m[k] for k in ["accuracy", "macro_f1", "mcc", "kappa", "mean_auc"]},
                         **{f"auc_{c}": v["auc"] for c, v in m["per_class"].items()}))
    print(f"[CV] completed {len(rows)}/{CFG['cv_folds']} folds")
    if not rows: return None
    df = pd.DataFrame(rows); df.to_csv(os.path.join(CFG["out_dir"], "table_cv_folds.csv"), index=False)
    print(df.describe().loc[["mean", "std"]] if len(df) > 1 else df)
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  7. ANALYSES: occlusion/Shapley, Grad-CAM, t-SNE, complexity, domain shift
# ══════════════════════════════════════════════════════════════════════════════
def load_primary(variant, seed):
    m = WaveSSMTriNet(variant).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(CFG["out_dir"], f"ckpt_{variant}_s{seed}.pt"), map_location=DEVICE))
    return m.eval()


@torch.no_grad()
def set_branch_means(model, loader, max_batches=40):
    sums = {n: 0.0 for n in model.branches}; cnt = 0
    for b, (x, _, _) in enumerate(loader):
        if b >= max_batches: break
        toks, _ = model.branch_tokens(x.to(DEVICE))
        for n in sums: sums[n] = sums[n] + toks[n].float().sum(0, keepdim=True)
        cnt += x.size(0)
    for n in sums: getattr(model, f"mean_{n}").copy_(sums[n] / cnt)


def occlusion_and_shapley(model, train_df, test_df):
    set_branch_means(model, make_loader(train_df.sample(min(1024, len(train_df)), random_state=0), False))
    loader = make_loader(test_df, False); names = list(model.branches)
    vals = {}
    for r in range(len(names) + 1):
        for present in combinations(names, r):
            drop = tuple(n for n in names if n not in present)
            c = n_ = 0
            with torch.no_grad():
                for x, y, _ in loader:
                    with torch.autocast("cuda", enabled=False):
                        o = model(x.to(DEVICE), drop=drop)
                    c += (o["prob"].argmax(1).cpu() == y).sum().item(); n_ += y.size(0)
            vals[frozenset(present)] = c / n_
    phi = shapley_values(vals, names)
    full = vals[frozenset(names)]
    out = dict(full_accuracy=full, coalitions={"+".join(sorted(k)) or "none": v for k, v in vals.items()},
               single_branch_removal_drop={n: full - vals[frozenset(set(names) - {n})] for n in names}, shapley_accuracy=phi)
    json.dump(out, open(os.path.join(CFG["out_dir"], "analysis_branch_occlusion_shapley.json"), "w"), indent=2)
    fig, ax = plt.subplots(figsize=(5, 4)); ax.bar(list(phi), [100 * v for v in phi.values()], color=PALETTE[:len(phi)])
    ax.set_ylabel("Shapley value of accuracy (pp)"); ax.set_title("Branch contribution (occlusion/Shapley)", fontweight="bold")
    fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_branch_shapley.png"), dpi=300); plt.close(fig)
    return out


def gradcam_all_branches(model, test_df, n_per_class=2):
    """Grad-CAM for each branch's feature map w.r.t. the predicted-class logit of the FULL model."""
    model.eval(); model.capture = True
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    hits, rows = {n: [] for n in model.branches}, []
    pick = pd.concat([g.sample(min(n_per_class, len(g)), random_state=0) for _, g in test_df.groupby("label")])
    for _, r in pick.iterrows():
        with Image.open(r.path) as im:
            x = build_transforms(False)(im.convert("RGB")).unsqueeze(0).to(DEVICE).requires_grad_(True)
        out = model(x); c = out["logits"].argmax(1).item()
        model.zero_grad(); out["logits"][0, c].backward()
        cams = {}
        for n, m in model._maps.items():
            if m.grad is None: continue
            w = m.grad.mean((2, 3), keepdim=True); cam = F.relu((w * m).sum(1, keepdim=True))
            cam = F.interpolate(cam, size=(CFG["img_size"],) * 2, mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
            cams[n] = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        img = (x[0].detach().cpu().permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
        rows.append((img, cams, int(r.label), c))
        if CFG["mask_root"]:                                            # optional pointing-game hit rate
            stem = os.path.splitext(os.path.basename(r.path))[0]
            for ext in (".png", ".jpg", ".tif"):
                mp = os.path.join(CFG["mask_root"], stem + ext)
                if os.path.exists(mp):
                    mk = np.asarray(Image.open(mp).convert("L").resize((CFG["img_size"],) * 2)) > 0
                    for n, cam in cams.items(): hits[n].append(bool(mk[np.unravel_index(cam.argmax(), cam.shape)]))
                    break
    model.capture = False
    names = list(model.branches)
    fig, axes = plt.subplots(len(rows), 1 + len(names), figsize=(2.6 * (1 + len(names)), 2.6 * len(rows)))
    axes = np.atleast_2d(axes)
    for i, (img, cams, yt, yp) in enumerate(rows):
        axes[i, 0].imshow(img); axes[i, 0].set_title(f"true={CFG['class_names'][yt]}\npred={CFG['class_names'][yp]}", fontsize=8)
        for j, n in enumerate(names, 1):
            axes[i, j].imshow(img); axes[i, j].imshow(cams.get(n, np.zeros_like(img[..., 0])), cmap="jet", alpha=0.45)
            if i == 0: axes[i, j].set_title(f"Grad-CAM: {n}", fontsize=8)
    for a in axes.ravel(): a.axis("off")
    fig.suptitle("Grad-CAM of each branch w.r.t. the full-model decision (qualitative; not a localisation claim)", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_gradcam_all_branches.png"), dpi=300); plt.close(fig)
    if CFG["mask_root"]:
        json.dump({n: (float(np.mean(h)) if h else None) for n, h in hits.items()}, open(os.path.join(CFG["out_dir"], "analysis_gradcam_hit_rate.json"), "w"), indent=2)


def embeddings(model, test_df):
    from sklearn.manifold import TSNE
    sub = pd.concat([g.sample(min(CFG["tsne_per_class"], len(g)), random_state=0) for _, g in test_df.groupby("label")]).reset_index(drop=True)
    P = predict(model, make_loader(sub, False), tta=False, with_fused=True)
    Z, y = P["fused"], P["label"]
    perp = min(30, max(5, len(y) // 4))
    ts = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(Z)   # sklearn default: 1000 iterations
    fig_embedding(ts, y, CFG["class_names"], os.path.join(CFG["out_dir"], "fig_tsne_balanced.png"),
                     f"t-SNE of fused features (stratified {CFG['tsne_per_class']}/class, perplexity {perp}, PCA init, seed 0)")
    try:
        import umap
        um = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Z)
        fig_embedding(um, y, CFG["class_names"], os.path.join(CFG["out_dir"], "fig_umap_balanced.png"), "UMAP of fused features (n_neighbors=15, min_dist=0.1)")
    except Exception as e:
        print("[UMAP] skipped:", e)


def scan_flops_estimate(L=196, D=192, S=16, layers=3):
    """Analytic cost of the custom scan + discretisation (elementwise ops are NOT counted by FlopCounterMode)."""
    steps = math.ceil(math.log2(L))
    per = L * D * S * (3 * steps + 6)        # 3 mul/add per scan step; exp, dB x, C h, reduction
    return layers * per


def complexity_and_speed(model):
    from torch.utils.flop_counter import FlopCounterMode
    model.eval(); x = torch.randn(1, 3, CFG["img_size"], CFG["img_size"], device=DEVICE)
    rep = {}
    with torch.no_grad():
        with FlopCounterMode(display=False) as fc: model(x)
        rep["total_flops_G_matmul_conv"] = fc.get_total_flops() / 1e9
        toks, _ = model.branch_tokens(x)
        for n, mod in (("wt", "wavelet_branch"), ("ssm", "ssm_branch"), ("cnn", "cnn_branch")):
            if n in model.branches:
                with FlopCounterMode(display=False) as f2: getattr(model, mod)(x)
                rep[f"{n}_flops_G"] = f2.get_total_flops() / 1e9
        with FlopCounterMode(display=False) as f3: z = model.fusion(toks)
        rep["fusion_flops_G"] = f3.get_total_flops() / 1e9
        with FlopCounterMode(display=False) as f4: model.head(z)
        rep["head_flops_G"] = f4.get_total_flops() / 1e9
    rep["ssm_scan_elementwise_flops_G_estimate"] = scan_flops_estimate() / 1e9 if "ssm" in model.branches else 0.0
    rep["total_params_M"] = sum(p.numel() for p in model.parameters()) / 1e6
    rep["trainable_params_M"] = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    rows = [(n, p.numel(), p.requires_grad) for n, p in model.named_parameters()]       # layer-wise table (comment 14)
    pd.DataFrame(rows, columns=["parameter", "numel", "trainable"]).to_csv(os.path.join(CFG["out_dir"], "table_layerwise_params.csv"), index=False)
    rep["module_params_M"] = {n: sum(p.numel() for p in m.parameters()) / 1e6 for n, m in model.named_children()}
    speed = {}
    for bs in (1, 32):
        xb = torch.randn(bs, 3, CFG["img_size"], CFG["img_size"], device=DEVICE)
        for amp in ([False, True] if USE_AMP else [False]):
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                for _ in range(2 if SMOKE else 20): model(xb)
                if USE_AMP: torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                ts = []
                for _ in range(3 if SMOKE else 100):
                    t0 = time.perf_counter(); model(xb)
                    if USE_AMP: torch.cuda.synchronize()
                    ts.append((time.perf_counter() - t0) * 1000)
            speed[f"bs{bs}_{'fp16' if amp else 'fp32'}"] = dict(
                latency_ms_mean=float(np.mean(ts)), latency_ms_sd=float(np.std(ts)), throughput_img_s=float(bs / (np.mean(ts) / 1000)),
                peak_mem_MB=float(torch.cuda.max_memory_allocated() / 2 ** 20) if USE_AMP else None)
    rep["speed"] = speed; rep["device"] = torch.cuda.get_device_name(0) if USE_AMP else "cpu"
    json.dump(rep, open(os.path.join(CFG["out_dir"], "analysis_complexity.json"), "w"), indent=2)
    return rep


def attention_entropy(model, loader, max_batches=10):
    """Evidence that TCAFG attention is non-degenerate (comment 12): entropy / log(N_k), 1.0 = uniform."""
    if not isinstance(model.fusion, TCAFG): return None
    ent = {}
    with torch.no_grad():
        for b, (x, _, _) in enumerate(loader):
            if b >= max_batches: break
            o = model(x.to(DEVICE), return_attn=True)
            for k, w in o["attn"].items():
                w = w.float().clamp_min(1e-12)
                ent.setdefault(k, []).append((-(w * w.log()).sum(-1) / math.log(w.size(-1))).mean().item())
    out = {k: float(np.mean(v)) for k, v in ent.items()}
    json.dump(out, open(os.path.join(CFG["out_dir"], "analysis_attention_entropy.json"), "w"), indent=2)
    return out


def domain_shift_table(manifest):
    """Acquisition-shift descriptors BRISC-test vs external, per class (comments 25, 28)."""
    a = manifest[manifest.role == "test"].sample(frac=1, random_state=0).groupby("label").head(300)
    b = manifest[manifest.role == "external"].sample(frac=1, random_state=0).groupby("label").head(300) if (manifest.role == "external").any() else None
    parts = [image_intensity_stats(a.path.tolist()).assign(set="brisc_test", label=a.label.values)]
    if b is not None: parts.append(image_intensity_stats(b.path.tolist()).assign(set="external", label=b.label.values))
    d = pd.concat(parts); d["class"] = d.label.map(dict(enumerate(CFG["class_names"])))
    t = d.groupby(["set", "class"])[["mean", "std", "bg_frac", "fg_mean", "p95"]].agg(["mean", "std"]); t.to_csv(os.path.join(CFG["out_dir"], "analysis_domain_shift_intensity.csv"))
    print(t)


# ══════════════════════════════════════════════════════════════════════════════
#  8. SMOKE-TEST DATA, REPRODUCIBILITY DUMP, MAIN
# ══════════════════════════════════════════════════════════════════════════════
def make_smoke_data():
    rng = np.random.RandomState(0)
    for root, splits, n in (("/tmp/smoke/brisc/classification_task", {"train": 10, "test": 4}, None), ("/tmp/smoke/ext/Training", None, 5)):
        for sp, cnt in (splits.items() if splits else [("", n)]):
            for c in CFG["class_names"]:
                d = os.path.join(root, sp, c) if sp else os.path.join(root, c); os.makedirs(d, exist_ok=True)
                for i in range(cnt):
                    a = (rng.rand(64, 64, 3) * 120 + 40 * CFG["class_names"].index(c)).clip(0, 255).astype(np.uint8)
                    Image.fromarray(a).save(os.path.join(d, f"{c}_{i}.jpg"))
    shutil_src = "/tmp/smoke/brisc/classification_task/test/glioma/glioma_0.jpg"       # planted train/test duplicate
    Image.open(shutil_src).save("/tmp/smoke/brisc/classification_task/train/glioma/dup_of_test.jpg")


def dump_reproducibility():
    json.dump({k: v for k, v in CFG.items()}, open(os.path.join(CFG["out_dir"], "run_config.json"), "w"), indent=2, default=str)
    json.dump(VARIANTS, open(os.path.join(CFG["out_dir"], "variants.json"), "w"), indent=2)
    with open(os.path.join(CFG["out_dir"], "environment.txt"), "w") as f:
        f.write(f"python {sys.version}\ntorch {torch.__version__}\ntimm {timm.__version__}\ndevice {DEVICE}\n")


# ══════════════════════════════════════════════════════════════════════════════
#  9. MULTI-GPU ORCHESTRATION (one worker process per GPU) + TIME BUDGET
# ══════════════════════════════════════════════════════════════════════════════
def job_list():
    """Priority order: first 3 seeds of every variant -> CV folds -> remaining seeds. A time-budget cut drops the tail."""
    seeds, variants = CFG["seeds"], CFG["variants"]
    jobs = [("main", v, s) for s in seeds[:3] for v in variants]
    if CFG["run_cv"]: jobs += [("cv", f) for f in range(1, CFG["cv_folds"] + 1)]
    jobs += [("main", v, s) for s in seeds[3:] for v in variants]
    return jobs


def job_name(j):
    return f"main_{j[1]}_s{j[2]}" if j[0] == "main" else f"cv_{j[1]}"


def claim(job):
    """Atomic cross-process job claim (O_EXCL file) -> dynamic load balancing between the GPU workers."""
    d = os.path.join(CFG["out_dir"], "claims"); os.makedirs(d, exist_ok=True)
    try:
        os.close(os.open(os.path.join(d, job_name(job)), os.O_CREAT | os.O_EXCL | os.O_WRONLY)); return True
    except FileExistsError:
        return False


def worker_main(manifest=None):
    global TRAIN_DEADLINE
    rank, world = int(os.environ.get("WAVESSM_WORKER", "0")), int(os.environ.get("WAVESSM_WORLD", "1"))
    TRAIN_DEADLINE = T0 + CFG["time_budget_h"] * 3600 - CFG["analysis_reserve_min"] * 60
    if not SMOKE: CFG["num_workers"] = max(2, (os.cpu_count() or 4) // world)
    if manifest is None: manifest = pd.read_csv(os.path.join(CFG["out_dir"], "split_manifest.csv"))
    print(f"[WORKER {rank}/{world}] device={DEVICE} " + (torch.cuda.get_device_name(0) if USE_AMP else "") +
          f"  loader_workers={CFG['num_workers']}  train deadline in {(TRAIN_DEADLINE - time.time()) / 3600:.2f} h", flush=True)
    dur = {}
    for job in job_list():
        if not claim(job): continue
        kind = job[0]
        est = dur.get("main") if kind == "main" else dur.get("cv", dur["main"] * (CFG["cv_epochs"] or CFG["epochs"]) / CFG["epochs"] if "main" in dur else None)
        if est and time.time() + 1.10 * est > TRAIN_DEADLINE:
            print(f"[BUDGET] worker {rank}: skipping {job_name(job)} (needs ~{est / 60:.0f} min, "
                  f"{max(0, TRAIN_DEADLINE - time.time()) / 60:.0f} min left for training)", flush=True); continue
        t = time.time()
        try:
            resumed = main_job(manifest, job[1], job[2]) if kind == "main" else cv_job(manifest, job[1])
            if not resumed: dur[kind] = time.time() - t
            print(f"[JOB DONE] {job_name(job)}  {(time.time() - t) / 60:.1f} min  (total elapsed {(time.time() - T0) / 3600:.2f} h)", flush=True)
        except BudgetExceeded as e:
            print(f"[BUDGET] {e}", flush=True)
        except Exception:
            print(f"[JOB FAILED] {job_name(job)}\n{traceback.format_exc()}", flush=True)
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    print(f"[WORKER {rank}] finished", flush=True)


def _script_path():
    """Path of this script, or (inside a notebook) the current cell's source written to a file."""
    if "__file__" in globals(): return os.path.abspath(__file__)
    try:
        src = get_ipython().user_ns["In"][-1]                           # noqa: F821  (the cell being executed)
    except Exception:
        return None
    if src.lstrip().startswith("%%"): src = src.split("\n", 1)[1]
    if "def worker_main" not in src or "WAVESSM-TRINET-SCRIPT" not in src: return None
    p = os.path.join(CFG["out_dir"], "_wavessm_script.py")
    with open(p, "w", encoding="utf-8") as f: f.write(src)
    return p


def parallel_world():
    n = int(os.environ.get("WAVESSM_FORCE_WORKERS", "0")) or (torch.cuda.device_count() if torch.cuda.is_available() else 0)
    return n if n >= 2 else 1


def _monitor(procs, paths):
    pos = [0] * len(procs)
    def drain():
        for i, p in enumerate(paths):
            with open(p, "rb") as f:
                f.seek(pos[i]); chunk = f.read()
            k = chunk.rfind(b"\n")
            if k < 0: continue
            pos[i] += k + 1
            for line in chunk[:k + 1].decode("utf-8", "replace").splitlines():
                if line.strip(): print(f"[gpu{i}] {line}", flush=True)
    while any(p.poll() is None for p in procs):
        drain(); time.sleep(5)
    drain()
    return [p.returncode for p in procs]


def run_jobs(manifest):
    shutil_rmtree = __import__("shutil").rmtree
    shutil_rmtree(os.path.join(CFG["out_dir"], "claims"), ignore_errors=True)
    n = parallel_world()
    path = _script_path() if n >= 2 else None
    if n >= 2 and path is None:
        print("[PARALLEL] could not locate this script's source (run it as ONE notebook cell or as a .py file) -> using a single GPU")
        n = 1
    if n < 2:
        print("[PARALLEL] single process"); worker_main(manifest); return
    print(f"[PARALLEL] launching {n} worker processes (one per GPU); jobs in priority order: {[job_name(j) for j in job_list()]}")
    procs, logs, fhs = [], [], []
    try:
        for r in range(n):
            env = dict(os.environ, WAVESSM_WORKER=str(r), WAVESSM_WORLD=str(n), WAVESSM_T0=str(T0), RUN_MODE=RUN_MODE)
            if torch.cuda.is_available(): env["CUDA_VISIBLE_DEVICES"] = str(r)
            lp = os.path.join(CFG["out_dir"], f"worker{r}.log"); fh = open(lp, "w")
            procs.append(subprocess.Popen([sys.executable, "-u", path], env=env, stdout=fh, stderr=subprocess.STDOUT)); logs.append(lp); fhs.append(fh)
            time.sleep(15 if (torch.cuda.is_available() and not SMOKE) else 1)    # stagger start-up (weight cache / IO)
        codes = _monitor(procs, logs)
    finally:
        for p in procs:
            if p.poll() is None: p.terminate()
        for fh in fhs: fh.close()
    print(f"[PARALLEL] workers exited with codes {codes}")
    if any(c != 0 for c in codes): print("[PARALLEL] WARNING: a worker crashed; see worker*.log. Completed runs are kept; re-run the cell to resume.")


def warm_pretrained_cache():
    """Download the timm weights once, up front (fails early and clearly without Internet)."""
    if not CFG["pretrained"]: return
    for b in sorted({vcfg(v, "backbone") for v in CFG["variants"]}):
        try: timm.create_model(b, pretrained=True, num_classes=0, global_pool="")
        except Exception as e: raise RuntimeError(f"Cannot load pretrained {b}: {e}. Turn Internet ON (or attach the weights).")
        print(f"[WEIGHTS] {b} pretrained weights available")


def main():
    if os.environ.get("WAVESSM_WORKER") is not None:
        worker_main(); return
    if SMOKE: make_smoke_data()
    dump_reproducibility(); test_haar_vs_pywt()
    print(f"[ENV] {torch.cuda.device_count() if torch.cuda.is_available() else 0} GPU(s): " +
          ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())) if torch.cuda.is_available() else "[ENV] no GPU")
    manifest = build_data()
    warm_pretrained_cache()
    run_jobs(manifest)
    print(f"\n[TIME] training phase finished after {(time.time() - T0) / 3600:.2f} h")
    runs = load_runs(); aggregate(runs)
    if CFG["run_cv"]: cv_aggregate()
    v, s = CFG["primary_variant"], CFG["seeds"][0]
    if CFG["run_analysis"] and os.path.exists(os.path.join(CFG["out_dir"], f"ckpt_{v}_s{s}.pt")):
        model = load_primary(v, s)
        te, tr = manifest[manifest.role == "test"].reset_index(drop=True), manifest[manifest.role == "train"]
        print("[ANALYSIS] attention entropy:", attention_entropy(model, make_loader(te, False)))
        print("[ANALYSIS] complexity/speed:", json.dumps(complexity_and_speed(model), indent=1)[:600])
        occlusion_and_shapley(model, tr, te); gradcam_all_branches(model, te); embeddings(model, te); domain_shift_table(manifest)
        hist = pd.read_csv(os.path.join(CFG["out_dir"], f"history_{v}_s{s}.csv"))
        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        ax[0].plot(hist.epoch, hist.train_loss, label="train"); ax[0].plot(hist.epoch, hist.val_loss, label="val (held-out)"); ax[0].legend(); ax[0].set_title("Loss")
        ax[1].plot(hist.epoch, 100 * hist.train_acc, label="train"); ax[1].plot(hist.epoch, 100 * hist.val_acc, label="val raw"); ax[1].plot(hist.epoch, 100 * hist.ema_val_acc, label="val EMA"); ax[1].legend(); ax[1].set_title("Accuracy (%)")
        ax[2].plot(np.load(os.path.join(CFG["out_dir"], f"lr_steps_{v}_s{s}.npy"))); ax[2].set_title("LR trajectory (per optimiser step)")
        fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig_training_curves_lr.png"), dpi=300); plt.close(fig)
    elif CFG["run_analysis"]:
        print(f"[ANALYSIS] skipped: checkpoint of the primary run {v}_s{s} does not exist")
    print(f"\n[DONE] outputs in {CFG['out_dir']}   total time {(time.time() - T0) / 3600:.2f} h")


if __name__ == "__main__":
    main()

[CONFIG] RUN_MODE=paper  seeds=[42, 43, 44, 45, 46]  epochs=50  variants=['full', 'softmax_head']  cv=True
[TEST] Haar DWT vs PyWavelets (LL, LH, HL, HH): [True, True, True, True]
[ENV] 2 GPU(s): Tesla T4, Tesla T4
[DATA] BRISC train=5000  test=1000  class counts test=[254, 306, 140, 300]
[DUP] near-duplicate pairs (cos>=0.985): train-train=153  test-test=26  train-test=176
[DUP] train images sharing a cluster with a test image: 159 (dropped)
[EXT] 5600 images  per-class counts (VERIFY against the paper's 5,600): [1400, 1400, 1400, 1400]
[EXT] images near-duplicating a BRISC image: 3863 (dropped)
                      glioma  meningioma  notumor  pituitary
role                                                        
dropped_dup_of_test       26         103        0         30
external                 267          95     1336         39
external_dropped_dup    1133        1305       64       1361
test                     254         306      140        300
train                   1010  

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

[WEIGHTS] efficientnet_b3 pretrained weights available
[PARALLEL] launching 2 worker processes (one per GPU); jobs in priority order: ['main_full_s42', 'main_softmax_head_s42', 'main_full_s43', 'main_softmax_head_s43', 'main_full_s44', 'main_softmax_head_s44', 'cv_1', 'cv_2', 'cv_3', 'cv_4', 'cv_5', 'main_full_s45', 'main_softmax_head_s45', 'main_full_s46', 'main_softmax_head_s46']
[gpu0] [CONFIG] RUN_MODE=paper  seeds=[42, 43, 44, 45, 46]  epochs=50  variants=['full', 'softmax_head']  cv=True
[gpu0] [WORKER 0/2] device=cuda Tesla T4  loader_workers=2  train deadline in 9.73 h
[gpu0] ======================================================================
[gpu0]   full_s42
[gpu0] ======================================================================
[gpu0] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[gpu1] [CONFIG] RUN_MODE=paper  seeds=[42, 43, 44, 45, 46]  epochs=50  variants=['full', 'softmax